In [ ]:
import json
from pprint import pprint
# with open(r'C:\Users\juani\Documents\Github\Abaqus_WELL_\wellClosure_axi.json') as f:
with open(r'C:\Users\hidalgo\Documents\GitHub\Abaqus_WELL_\wellClosure_axi.json') as f:
    data = json.load(f)

pprint(data["Lithology"])
print("\n\n")
data["AnalysisData"]["Top"] = 3200
data["AnalysisData"]["Bottom"] = 4250

materials = dict()

def process_lithology(data):
    top = data["AnalysisData"]["Top"]
    bottom = data["AnalysisData"]["Bottom"]
    t_depths = set((top, bottom))
    filtered_layers = []
    filtered_rocks = {}
    lithology = data["Lithology"]
    json_rocks = data["Rocks"]
    
    for i, layer in enumerate(lithology):
        lname = "LAYER_%.2d" % (i+1)
        if layer["Bottom"] <= top or layer["Top"] >= bottom:
            continue
        if layer["Top"] < top and layer["Bottom"] > top:
            rock_mat = json_rocks[layer["Rock"]].copy()
            rock_mat["Name"] = lname + "_Material"
            filtered_rocks[rock_mat["Name"]] = rock_mat
            filtered_layers.append({**layer, "Name": lname, "Top": top, "Material": rock_mat["Name"]})
        elif layer["Top"] < bottom and layer["Bottom"] > bottom:
            rock_mat = json_rocks[layer["Rock"]].copy()
            rock_mat["Name"] = lname + "_Material"
            filtered_rocks[rock_mat["Name"]] = rock_mat
            filtered_layers.append({**layer, "Name": lname, "Bottom": bottom, "Material": rock_mat["Name"]})
        else:
            rock_mat = json_rocks[layer["Rock"]].copy()
            rock_mat["Name"] = lname + "_Material"
            filtered_rocks[rock_mat["Name"]] = rock_mat
            filtered_layers.append({**layer, "Name": lname, "Material": rock_mat["Name"]})
        t_depths.add(layer["Top"])
        t_depths.add(layer["Bottom"])

    return filtered_layers, t_depths, filtered_rocks


filtered_layers, t_depths, filtered_rocks = process_lithology(data) 
pprint(filtered_layers)
pprint(filtered_rocks)
print(sorted(t_depths))

# t_depths = set([x for x in depths if A_depths[0]<= x <= A_depths[1]])
# t_depths.add(A_depths[0])
# t_depths.add(A_depths[1])


[{'Bottom': 2500.0, 'Rock': 'SHALE', 'Top': 2000.0},
 {'Bottom': 3600.0, 'Rock': 'SANDSTONE', 'Top': 2500.0},
 {'Bottom': 4150.0, 'Rock': 'HALITE', 'Top': 3600.0},
 {'Bottom': 4250.0, 'Rock': 'CARNALLITE', 'Top': 4150.0},
 {'Bottom': 4400.0, 'Rock': 'HALITE_DM', 'Top': 4250.0},
 {'Bottom': 4500.0, 'Rock': 'TACHYHYDRITE', 'Top': 4400.0},
 {'Bottom': 5000.0, 'Rock': 'HALITE', 'Top': 4500.0},
 {'Bottom': 5200.0, 'Rock': 'ANHYDRITE', 'Top': 5000.0},
 {'Bottom': 5500.0, 'Rock': 'CARBONATE', 'Top': 5200.0}]



[{'Bottom': 3600.0,
  'Material': 'LAYER_02_Material',
  'Name': 'LAYER_02',
  'Rock': 'SANDSTONE',
  'Top': 3200},
 {'Bottom': 4150.0,
  'Material': 'LAYER_03_Material',
  'Name': 'LAYER_03',
  'Rock': 'HALITE',
  'Top': 3600.0},
 {'Bottom': 4250.0,
  'Material': 'LAYER_04_Material',
  'Name': 'LAYER_04',
  'Rock': 'CARNALLITE',
  'Top': 4150.0}]
{'LAYER_02_Material': {'ElasticParameters': {'Density': 2300.0,
                                             'Poisson': 0.25,
              

In [11]:
import json
from pprint import pprint
# with open(r'C:\Users\juani\Documents\Github\Abaqus_WELL_\wellClosure_axi.json') as f:
with open(r'C:\Users\hidalgo\Documents\GitHub\Abaqus_WELL_\wellClosure_axi.json') as f:
    data = json.load(f)

# pprint(data["Lithology"])
# print("\n\n")
data["AnalysisData"]["Top"] = 3200
data["AnalysisData"]["Bottom"] = 4250

# materials = dict()

def process_lithology(data):
    # 1. Pega os limites globais do modelo (Ex: 3200 e 4250)
    global_top = data["AnalysisData"]["Top"]
    global_bottom = data["AnalysisData"]["Bottom"]
    
    # 2. Inicia o set de profundidades já com os limites globais
    # t_depths = set([global_top, global_bottom])
    t_depths = set()
    
    filtered_layers = []
    filtered_rocks = {}
    
    lithology = data["Lithology"]
    json_rocks = data["Rocks"]
    
    # Usamos um contador manual para nomear as camadas sequencialmente (LAYER_01, LAYER_02...)
    layer_counter = 1 
    
    for layer in lithology:
        l_top = layer["Top"]
        l_bottom = layer["Bottom"]
        
        # Ignora as rochas que estão completamente fora do domínio do poço
        if l_bottom <= global_top or l_top >= global_bottom:
            continue
            
        # A MÁGICA ACONTECE AQUI (Clipping):
        # Se o topo da rocha for menor que o topo do poço, ele corta no topo do poço.
        # Se a base da rocha for maior que a base do poço, ele corta na base do poço.
        clipped_top = max(l_top, global_top)
        clipped_bottom = min(l_bottom, global_bottom)
        
        # Formata o nome da camada (ex: LAYER_01)
        lname = "LAYER_%02d" % layer_counter
        layer_counter += 1
        
        # Prepara o material da rocha
        rock_name = layer["Rock"]
        rock_mat = json_rocks[rock_name].copy()
        mat_name = lname + "_Material"
        rock_mat["Name"] = mat_name
        filtered_rocks[mat_name] = rock_mat
        
        # Cria a nova camada com os valores CORTADOS e substitui/adiciona as chaves necessárias
        new_layer = layer.copy()
        new_layer["Name"] = lname
        new_layer["Material"] = mat_name
        new_layer["Top"] = clipped_top
        new_layer["Bottom"] = clipped_bottom
        
        filtered_layers.append(new_layer)
        
        # Adiciona as profundidades cortadas ao set (para particionar a geometria no Abaqus depois)
        t_depths.add(clipped_top)
        t_depths.add(clipped_bottom)
        
    # Remove as fronteiras globais do set, deixando apenas os cortes intermediários
    t_depths.discard(global_top)
    t_depths.discard(global_bottom)

    return filtered_layers, t_depths, filtered_rocks

filtered_layers, t_depths, filtered_rocks = process_lithology(data) 

pprint(filtered_layers)

pprint(filtered_rocks)

print(sorted(t_depths))

[{'Bottom': 3600.0,
  'Material': 'LAYER_01_Material',
  'Name': 'LAYER_01',
  'Rock': 'SANDSTONE',
  'Top': 3200},
 {'Bottom': 4150.0,
  'Material': 'LAYER_02_Material',
  'Name': 'LAYER_02',
  'Rock': 'HALITE',
  'Top': 3600.0},
 {'Bottom': 4250.0,
  'Material': 'LAYER_03_Material',
  'Name': 'LAYER_03',
  'Rock': 'CARNALLITE',
  'Top': 4150.0}]
{'LAYER_01_Material': {'ElasticParameters': {'Density': 2300.0,
                                             'Poisson': 0.25,
                                             'Young': 24.062023},
                       'Elasticity': 'ISOTROPIC',
                       'Law': 'MOHR_COULOMB',
                       'MohrCoulombParameters': {'Cohesion': 20.0017,
                                                 'DilatancyAngle': 7.5,
                                                 'FrictionAngle': 30.0,
                                                 'UltimateTraction': 0.0},
                       'Name': 'LAYER_01_Material',
                     